# Tahap 1: Membangun Case Base
**Mata Kuliah:** Penalaran Komputer — SubCPMK-3 Case-Based Reasoning  
**Tujuan:** Scraping dan preprocessing corpus putusan dari Direktori MA RI  

### Langkah:
1. Setup & Install Dependencies
2. Import & Konfigurasi
3. Setup Logging
4. Scraping Metadata Putusan dari MA RI
5. Download PDF Putusan
6. Konversi PDF → Plain Text
7. Cleaning & Normalisasi Teks
8. Pipeline Utama
9. Validasi Akhir & Laporan
10. Preview Hasil

## Cell 1 — Install Dependencies

In [ ]:
# Install semua library yang dibutuhkan
!pip install requests beautifulsoup4 pdfminer.six tqdm lxml undetected-chromedriver selenium
!pip install pdf2image pytesseract pillow

# Install poppler via conda jika tersedia, atau via pip wheel (Windows)
import subprocess, sys

def install_poppler():
    """Coba install poppler otomatis."""
    # Coba via conda (paling reliable)
    try:
        result = subprocess.run(
            ["conda", "install", "-y", "-c", "conda-forge", "poppler"],
            capture_output=True, text=True, timeout=120
        )
        if result.returncode == 0:
            print("✅ Poppler berhasil diinstall via conda")
            return "conda"
    except Exception:
        pass

    # Coba via pip wheel (Windows only)
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "poppler-utils"])
        print("✅ Poppler berhasil diinstall via pip")
        return "pip"
    except Exception:
        pass

    print("⚠️  Poppler tidak bisa diinstall otomatis.")
    print("   Download manual dari:")
    print("   https://github.com/oschwartz10612/poppler-windows/releases")
    print("   Extract lalu set POPPLER_PATH di Cell 2 (CONFIG)")
    return None

install_poppler()

## Cell 2 — Import & Konfigurasi

In [1]:
import os
import re
import time
import json
import shutil
import logging
import unicodedata
import fitz
from pathlib import Path
from datetime import datetime
from io import StringIO

import requests
from bs4 import BeautifulSoup
from tqdm import tqdm

from pdfminer.high_level import extract_text
from pdfminer.layout import LAParams

print("✅ Semua library berhasil diimport")

✅ Semua library berhasil diimport


In [2]:
# ============================================================
# KONFIGURASI — Sesuaikan dengan kebutuhan tim
# ============================================================
CONFIG = {
    # Jenis perkara yang dipilih
    "JENIS_PERKARA"     : "Pidana Khusus Anak",

    # Keyword pencarian di MA RI
    "KEYWORD_PENCARIAN" : "pidana khusus anak",

    # Jumlah minimum dokumen (sesuai ketentuan tugas)
    "MIN_DOKUMEN"       : 50,

    # Jumlah dokumen target scraping (lebih sebagai buffer)
    "TARGET_DOKUMEN"    : 100,

    # Delay antar request (detik)
    "DELAY_REQUEST"     : 2,

    # ── Konfigurasi OCR ─────────────────────────────────────────────
    # Sesuaikan path Tesseract jika tidak ada di PATH Windows
    # Download: https://github.com/UB-Mannheim/tesseract/wiki
    "TESSERACT_PATH"    : r"C:\Program Files\Tesseract-OCR\tesseract.exe",

    # Sesuaikan path Poppler bin jika tidak ada di PATH Windows
    # Download: https://github.com/oschwartz10612/poppler-windows/releases
    # Isi dengan path folder bin-nya, contoh: r"C:\poppler\Library\bin"
    # Kosongkan ("") jika sudah ada di PATH atau install via conda
    "POPPLER_PATH"      : r"C:\Users\PC-LABIT2\Downloads\Release-26.02.0-0\poppler-26.02.0\Library\bin",

    # DPI untuk OCR — 200 bagus, 150 lebih cepat, 300 paling akurat
    "OCR_DPI"           : 200,

    # Bahasa OCR — "ind" untuk Indonesia, "ind+eng" untuk campuran
    "OCR_LANG"          : "ind",

    # Persentase minimum isi putusan yang harus tersedia
    "MIN_VALIDITAS"     : 0.05,    # sangat longgar untuk PDF scan

    # Minimum jumlah kata agar dokumen dianggap valid
    "MIN_KATA"          : 50,      # PDF scan OCR hasilnya lebih sedikit kata
}

# ============================================================
# SETUP DIREKTORI
# ============================================================
BASE_DIR = Path(".").resolve()
DATA_RAW = BASE_DIR / "data" / "raw"
DATA_PDF = BASE_DIR / "data" / "pdf"
LOGS_DIR = BASE_DIR / "logs"

for d in [DATA_RAW, DATA_PDF, LOGS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("📁 Direktori berhasil dibuat:")
print(f"   Raw text : {DATA_RAW}")
print(f"   PDF      : {DATA_PDF}")
print(f"   Logs     : {LOGS_DIR}")

📁 Direktori berhasil dibuat:
   Raw text : D:\amad\Paper\Tugas\PK\Penalaran-Komputer-Sub-3\STEP 1\data\raw
   PDF      : D:\amad\Paper\Tugas\PK\Penalaran-Komputer-Sub-3\STEP 1\data\pdf
   Logs     : D:\amad\Paper\Tugas\PK\Penalaran-Komputer-Sub-3\STEP 1\logs


## Cell 3 — Setup Logging

In [3]:
log_file = LOGS_DIR / "cleaning.log"

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    handlers=[
        logging.FileHandler(log_file, encoding="utf-8"),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)
logger.info(f"=== Sesi dimulai — Perkara: {CONFIG['JENIS_PERKARA']} ===")
print(f"📝 Log disimpan di: {log_file}")

2026-06-13 21:53:48,291 | INFO | === Sesi dimulai — Perkara: Pidana Khusus Anak ===


📝 Log disimpan di: D:\amad\Paper\Tugas\PK\Penalaran-Komputer-Sub-3\STEP 1\logs\cleaning.log


## Cell 4 — Scraping Metadata Putusan dari MA RI

> **URL Direktori:** https://putusan3.mahkamahagung.go.id/  
> Menggunakan `undetected-chromedriver` untuk bypass Cloudflare.  
> `headless=False` wajib agar tidak diblokir.

In [4]:
import subprocess, sys
import undetected_chromedriver as uc
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait

print("✅ Selenium & undetected-chromedriver siap")

# ============================================================
# FUNGSI UTILITAS SCRAPING
# (didefinisikan di sini agar tersedia di cell-cell berikutnya)
# ============================================================

def tunggu_konten(driver, timeout=30):
    """Tunggu sampai Cloudflare selesai dan konten MA RI muncul."""
    start = time.time()
    while time.time() - start < timeout:
        try:
            src   = driver.page_source
            title = driver.title.lower()
            if any(x in src for x in [
                "lds-ring", "ray-id", "cf-browser-verification", "checking your browser"
            ]):
                print("   ⏳ Cloudflare aktif, tunggu...")
                time.sleep(4)
                continue
            if any(x in src.lower() for x in ["putusan", "perkara", "mahkamah", "pengadilan"]):
                return True
            time.sleep(2)
        except Exception as e:
            print(f"   ⚠️  Window error: {e}")
            return False
    return False


def buat_driver(download_dir: Path = None):
    """
    Buat Chrome driver.
    Jika download_dir diberikan, browser akan auto-download ke folder tersebut.
    """
    opts = uc.ChromeOptions()
    opts.add_argument("--window-size=1920,1080")
    opts.add_argument("--lang=id-ID")
    opts.add_argument("--no-first-run")
    opts.add_argument("--disable-popup-blocking")

    if download_dir:
        prefs = {
            "download.default_directory"        : str(download_dir),
            "download.prompt_for_download"      : False,
            "download.directory_upgrade"        : True,
            "plugins.always_open_pdf_externally": True,
            "safebrowsing.enabled"              : False,
        }
        opts.add_experimental_option("prefs", prefs)

    driver = uc.Chrome(
        options    = opts,
        headless   = False,   # HARUS False agar Cloudflare tidak blokir
        keep_alive = True,
    )
    time.sleep(3)
    return driver


def parse_putusan(html: str, base_url="https://putusan3.mahkamahagung.go.id") -> list:
    """Parse HTML halaman hasil pencarian, kumpulkan link & metadata putusan."""
    soup  = BeautifulSoup(html, "lxml")
    hasil = []
    seen  = set()

    for pola in [re.compile(r"/direktori/putusan/[a-z0-9]"), re.compile(r"/putusan/[a-z0-9]")]:
        links = soup.find_all("a", href=pola)
        if links:
            break

    for lnk in links:
        href = lnk.get("href", "")
        teks = lnk.get_text(" ", strip=True)
        if not href or href in seen:
            continue
        seen.add(href)
        full_url = base_url + href if href.startswith("/") else href

        m_no = re.search(r"\d+\s*/\s*[A-Z][a-zA-Z.]+\s*/\s*\d{4}", teks + " " + href)
        m_tg = re.search(
            r"\d{4}-\d{2}-\d{2}|\d{1,2}[\s\-/](Januari|Februari|Maret|April|Mei|"
            r"Juni|Juli|Agustus|September|Oktober|November|Desember)[\s\-/]\d{4}",
            teks, re.I
        )
        hasil.append({
            "judul"      : teks[:150],
            "no_perkara" : m_no.group(0).strip() if m_no else "",
            "tanggal"    : m_tg.group(0).strip() if m_tg else "",
            "pengadilan" : "",
            "url_detail" : full_url,
        })
    return hasil


print("✅ Fungsi utilitas scraping siap")

✅ Selenium & undetected-chromedriver siap
✅ Fungsi utilitas scraping siap


In [5]:
# ============================================================
# MAIN SCRAPING — Buka browser & kumpulkan metadata putusan
# ============================================================
BASE_MA = "https://putusan3.mahkamahagung.go.id"
SEARCH  = f"{BASE_MA}/search.html"

print("🚀 Membuka browser Chrome...")
driver = buat_driver()

daftar_putusan = []

try:
    # Warmup homepage
    print("🏠 Membuka homepage...")
    driver.get(BASE_MA)
    ok = tunggu_konten(driver, timeout=35)
    print(f"   Homepage: {'✅ OK' if ok else '⚠️ Cloudflare'} — {driver.title[:50]}")
    time.sleep(3)

    halaman = 1
    keyword = CONFIG["KEYWORD_PENCARIAN"]
    target  = CONFIG["TARGET_DOKUMEN"]
    print(f"\n🔍 Mencari: '{keyword}' | Target: {target} dokumen")

    while len(daftar_putusan) < target:
        url = f"https://putusan3.mahkamahagung.go.id/search.html?court=099780PN75&cat=738477533355663d2fb5ec8476ccf6df&jenis_doc=putusan&obf=TOTAL_VIEW&obm=desc&page={halaman}"
        print(f"\n   📄 Halaman {halaman} → {url}")
        driver.get(url)

        ok = tunggu_konten(driver, timeout=30)
        if not ok:
            dump = LOGS_DIR / f"debug_hal{halaman}.html"
            dump.write_text(driver.page_source, encoding="utf-8")
            print(f"   ❌ Gagal load — HTML dump: {dump}")
            break

        time.sleep(CONFIG["DELAY_REQUEST"])
        hasil = parse_putusan(driver.page_source)

        if not hasil:
            dump = LOGS_DIR / f"debug_hal{halaman}_noresult.html"
            dump.write_text(driver.page_source, encoding="utf-8")
            print(f"   ⚠️  Halaman {halaman}: 0 putusan ditemukan, berhenti")
            print(f"   HTML dump: {dump}")
            break

        daftar_putusan.extend(hasil)
        print(f"   ✅ +{len(hasil)} putusan → total {len(daftar_putusan)}")
        halaman += 1
        time.sleep(CONFIG["DELAY_REQUEST"])

    daftar_putusan = daftar_putusan[:target]

except Exception as e:
    import traceback
    print(f"\n❌ Error: {e}")
    traceback.print_exc()

# Simpan metadata awal
if daftar_putusan:
    meta_file = BASE_DIR / "data" / "metadata_raw.json"
    meta_file.parent.mkdir(parents=True, exist_ok=True)
    with open(meta_file, "w", encoding="utf-8") as f:
        json.dump(daftar_putusan, f, ensure_ascii=False, indent=2)

    print(f"\n✅ {len(daftar_putusan)} putusan berhasil di-scrape")
    print(f"💾 Metadata: {meta_file}")
    print("\n📋 Preview 3 pertama:")
    for i, p in enumerate(daftar_putusan[:3], 1):
        print(f"  [{i}] {p['judul'][:70]}")
        print(f"       No     : {p['no_perkara']}")
        print(f"       URL    : {p['url_detail']}")
else:
    print("\n" + "="*55)
    print("  ⚠️  SCRAPING GAGAL")
    print("  Kemungkinan Cloudflare terlalu ketat.")
    print("  Coba jalankan ulang cell ini.")
    print("="*55)
    try:
        driver.quit()
    except:
        pass

🚀 Membuka browser Chrome...


2026-06-13 21:53:50,385 | INFO | patching driver executable C:\Users\PC-LABIT2\appdata\roaming\undetected_chromedriver\undetected_chromedriver.exe


🏠 Membuka homepage...
   ⏳ Cloudflare aktif, tunggu...
   Homepage: ✅ OK — Direktori Putusan

🔍 Mencari: 'pidana khusus anak' | Target: 100 dokumen

   📄 Halaman 1 → https://putusan3.mahkamahagung.go.id/search.html?court=099780PN75&cat=738477533355663d2fb5ec8476ccf6df&jenis_doc=putusan&obf=TOTAL_VIEW&obm=desc&page=1
   ✅ +23 putusan → total 23

   📄 Halaman 2 → https://putusan3.mahkamahagung.go.id/search.html?court=099780PN75&cat=738477533355663d2fb5ec8476ccf6df&jenis_doc=putusan&obf=TOTAL_VIEW&obm=desc&page=2
   ✅ +23 putusan → total 46

   📄 Halaman 3 → https://putusan3.mahkamahagung.go.id/search.html?court=099780PN75&cat=738477533355663d2fb5ec8476ccf6df&jenis_doc=putusan&obf=TOTAL_VIEW&obm=desc&page=3
   ✅ +23 putusan → total 69

   📄 Halaman 4 → https://putusan3.mahkamahagung.go.id/search.html?court=099780PN75&cat=738477533355663d2fb5ec8476ccf6df&jenis_doc=putusan&obf=TOTAL_VIEW&obm=desc&page=4
   ✅ +23 putusan → total 92

   📄 Halaman 5 → https://putusan3.mahkamahagung.go.id/searc

## Cell 5 — Download PDF Putusan

> Mencari link PDF di setiap halaman detail putusan.  
> Putusan tanpa lampiran PDF **di-skip** dan **tidak masuk hitungan kuota**.

**Catatan struktur halaman MA RI:**
```html
<a href="https://putusan3.mahkamahagung.go.id/direktori/download_file/{hash}/pdf/{hash}">
  712/PID.SUS/2018/PT_SBY.pdf
</a>
```
Href adalah URL hash — deteksi dilakukan dari **teks anchor** yang diakhiri `.pdf`.

In [6]:
# ============================================================
# FUNGSI DOWNLOAD PDF
# ============================================================

def cari_link_pdf(driver, url_detail: str):
    """
    Buka halaman detail putusan, cari link PDF di bagian Lampiran.

    Struktur MA RI: href adalah URL hash, teks anchor yang berformat PDF:
      <a href="https://putusan3.../direktori/download_file/.../pdf/...">
        712/PID.SUS/2018/PT_SBY.pdf
      </a>

    Strategi utama : cocokkan dari TEKS anchor yang diakhiri .pdf
    Fallback       : regex di page source

    Returns: URL href jika ditemukan, None jika tidak ada PDF.
    """
    driver.get(url_detail)
    tunggu_konten(driver, timeout=25)
    time.sleep(3)

    # Strategi 1: Selenium — teks anchor diakhiri .pdf
    semua_link = driver.find_elements(By.TAG_NAME, "a")
    for link in semua_link:
        try:
            teks = (link.text or "").strip()
            href = (link.get_attribute("href") or "").strip()
            if (teks.lower().endswith(".pdf")
                    and href
                    and ("download" in href or "direktori" in href)):
                return href
        except Exception:
            continue

    # Strategi 2: Fallback regex di page source
    src   = driver.page_source
    match = re.search(
        r'href=["\'\']([^\"\'\']+)["\'\'][^>]*>\s*[^<]*\.pdf\s*<',
        src, re.IGNORECASE
    )
    if match:
        return match.group(1)

    return None


def get_pdf_files(folder: Path) -> set:
    """Set nama file PDF yang ada di folder (untuk deteksi file baru)."""
    return {f.name for f in folder.glob("*.pdf")}


print("✅ Fungsi download PDF siap")

✅ Fungsi download PDF siap


In [7]:
# ============================================================
# SETUP DRIVER BARU DENGAN FOLDER DOWNLOAD
# ============================================================

# Tutup driver scraping jika masih aktif
try:
    driver.quit()
    time.sleep(2)
    print("🔒 Driver scraping ditutup")
except Exception:
    pass

DOWNLOAD_DIR = BASE_DIR / "data" / "pdf"
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
print(f"📁 Folder download: {DOWNLOAD_DIR}")

print("🚀 Membuka browser untuk download...")
driver_dl = buat_driver(download_dir=DOWNLOAD_DIR)

# Warmup
print("🏠 Warmup homepage...")
driver_dl.get("https://putusan3.mahkamahagung.go.id")
tunggu_konten(driver_dl, timeout=35)
time.sleep(3)
print(f"   OK: {driver_dl.title[:60]}")

# ============================================================
# LOOP DOWNLOAD
# ============================================================
print(f"\n⬇️  Memulai download untuk {len(daftar_putusan)} entri...\n")

daftar_dengan_pdf = []
n_berhasil       = 0
n_sudah_ada      = 0
n_skip_tanpa_pdf = 0
n_gagal_download = 0

for idx, putusan in enumerate(tqdm(daftar_putusan, desc="Download PDF"), start=1):
    case_id     = f"case_{idx:03d}"
    path_target = DOWNLOAD_DIR / f"{case_id}.pdf"
    putusan["case_id"] = case_id

    # Skip jika sudah ada
    if path_target.exists() and path_target.stat().st_size > 5000:
        logger.info(f"{case_id}: sudah ada, skip")
        putusan["path_pdf"]   = str(path_target)
        putusan["status_pdf"] = "sudah_ada"
        daftar_dengan_pdf.append(putusan)
        n_sudah_ada += 1
        continue

    # Cari link PDF
    pdf_url = cari_link_pdf(driver_dl, putusan["url_detail"])

    # Tidak ada PDF → skip, tidak masuk kuota
    if not pdf_url:
        logger.warning(f"{case_id}: tidak ada lampiran PDF → di-skip")
        putusan["path_pdf"]   = ""
        putusan["status_pdf"] = "skip_tanpa_pdf"
        daftar_dengan_pdf.append(putusan)
        n_skip_tanpa_pdf += 1
        continue

    logger.info(f"{case_id}: link PDF → {pdf_url}")

    # Catat PDF sebelum navigasi
    pdf_sebelum = get_pdf_files(DOWNLOAD_DIR)

    # Navigasi ke URL PDF (auto-download)
    driver_dl.get(pdf_url)
    time.sleep(2)

    # Tunggu file baru muncul (max 60 detik)
    pdf_baru = None
    for _ in range(60):
        time.sleep(1)
        pdf_sekarang = get_pdf_files(DOWNLOAD_DIR)
        pdf_baru_set = pdf_sekarang - pdf_sebelum
        pdf_baru_set = {f for f in pdf_baru_set if not f.endswith(".crdownload")}
        if pdf_baru_set:
            pdf_baru = DOWNLOAD_DIR / sorted(pdf_baru_set)[-1]
            break

    if pdf_baru and pdf_baru.exists() and pdf_baru.stat().st_size > 5000:
        shutil.move(str(pdf_baru), str(path_target))
        ukuran_kb = path_target.stat().st_size / 1024
        logger.info(f"{case_id}: ✅ {ukuran_kb:.0f} KB")
        putusan["path_pdf"]   = str(path_target)
        putusan["status_pdf"] = "berhasil"
        n_berhasil += 1
    else:
        logger.warning(f"{case_id}: timeout atau file terlalu kecil")
        putusan["path_pdf"]   = ""
        putusan["status_pdf"] = "gagal_download"
        n_gagal_download += 1

    putusan["pdf_url_used"] = pdf_url
    daftar_dengan_pdf.append(putusan)
    time.sleep(CONFIG["DELAY_REQUEST"])

# Tutup browser
try:
    driver_dl.quit()
    print("\n🔒 Browser ditutup")
except Exception:
    pass

# Simpan metadata
meta_pdf_file = BASE_DIR / "data" / "metadata_with_pdf.json"
with open(meta_pdf_file, "w", encoding="utf-8") as f:
    json.dump(daftar_dengan_pdf, f, ensure_ascii=False, indent=2)

# ── REKAP ────────────────────────────────────────────────────
total_pdf_folder = len(list(DOWNLOAD_DIR.glob("*.pdf")))
masuk_kuota      = n_berhasil + n_sudah_ada   # skip_tanpa_pdf TIDAK ikut

print(f"\n{'='*55}")
print(f"       REKAP DOWNLOAD PDF")
print(f"{'='*55}")
print(f"  ✅ Berhasil download   : {n_berhasil}")
print(f"  ⏭️  Sudah ada (skip)    : {n_sudah_ada}")
print(f"  🚫 Tidak ada PDF*      : {n_skip_tanpa_pdf}  ← tidak masuk kuota")
print(f"  ❌ Gagal/timeout       : {n_gagal_download}")
print(f"{'='*55}")
print(f"  📄 Masuk hitungan kuota: {masuk_kuota} / {CONFIG['MIN_DOKUMEN']} minimum")
print(f"  📁 Total PDF di folder : {total_pdf_folder}")
print(f"{'='*55}")
print(f"  * Putusan tanpa lampiran PDF di-skip")
print(f"    dan tidak dihitung dalam kuota minimum.")

if masuk_kuota >= CONFIG["MIN_DOKUMEN"]:
    print(f"\n✅ Kuota terpenuhi — lanjut ke Cell 6!")
else:
    kekurangan = CONFIG["MIN_DOKUMEN"] - masuk_kuota
    print(f"\n⚠️  Kurang {kekurangan} dokumen.")
    print(f"   Tambah TARGET_DOKUMEN di CONFIG atau coba keyword lain.")

print(f"\n💾 Metadata: {meta_pdf_file}")

🔒 Driver scraping ditutup
📁 Folder download: D:\amad\Paper\Tugas\PK\Penalaran-Komputer-Sub-3\STEP 1\data\pdf
🚀 Membuka browser untuk download...


2026-06-13 21:54:56,101 | INFO | patching driver executable C:\Users\PC-LABIT2\appdata\roaming\undetected_chromedriver\undetected_chromedriver.exe


🏠 Warmup homepage...
   ⏳ Cloudflare aktif, tunggu...
   OK: Direktori Putusan

⬇️  Memulai download untuk 100 entri...



Download PDF:   0%|          | 0/100 [00:00<?, ?it/s]2026-06-13 21:55:10,339 | INFO | case_001: sudah ada, skip
2026-06-13 21:55:10,340 | INFO | case_002: sudah ada, skip
2026-06-13 21:55:10,340 | INFO | case_003: sudah ada, skip
2026-06-13 21:55:10,341 | INFO | case_004: sudah ada, skip
2026-06-13 21:55:10,342 | INFO | case_005: sudah ada, skip
2026-06-13 21:55:10,343 | INFO | case_006: sudah ada, skip
2026-06-13 21:55:10,343 | INFO | case_007: sudah ada, skip
2026-06-13 21:55:10,344 | INFO | case_008: sudah ada, skip
2026-06-13 21:55:10,345 | INFO | case_009: sudah ada, skip
2026-06-13 21:55:10,346 | INFO | case_010: sudah ada, skip
2026-06-13 21:55:10,347 | INFO | case_011: sudah ada, skip
2026-06-13 21:55:10,348 | INFO | case_012: sudah ada, skip
2026-06-13 21:55:10,348 | INFO | case_013: sudah ada, skip
2026-06-13 21:55:10,349 | INFO | case_014: sudah ada, skip
2026-06-13 21:55:10,350 | INFO | case_015: sudah ada, skip
2026-06-13 21:55:10,350 | INFO | case_016: sudah ada, skip
202


🔒 Browser ditutup

       REKAP DOWNLOAD PDF
  ✅ Berhasil download   : 43
  ⏭️  Sudah ada (skip)    : 40
  🚫 Tidak ada PDF*      : 15  ← tidak masuk kuota
  ❌ Gagal/timeout       : 2
  📄 Masuk hitungan kuota: 83 / 50 minimum
  📁 Total PDF di folder : 85
  * Putusan tanpa lampiran PDF di-skip
    dan tidak dihitung dalam kuota minimum.

✅ Kuota terpenuhi — lanjut ke Cell 6!

💾 Metadata: D:\amad\Paper\Tugas\PK\Penalaran-Komputer-Sub-3\STEP 1\data\metadata_with_pdf.json


In [8]:
# ============================================================
# CELL TAMBAHAN — Filter dokumen tidak relevan + jalankan pipeline
# Jalankan ini setelah Cell 5 (download) selesai
# ============================================================
import json
from pathlib import Path

# Load metadata yang sudah ada
meta_file = BASE_DIR / "data" / "metadata_with_pdf.json"
with open(meta_file, encoding="utf-8") as f:
    daftar_dengan_pdf = json.load(f)

print(f"Total sebelum filter : {len(daftar_dengan_pdf)}")

# Filter: hanya ambil yang:
# 1. status_pdf = berhasil
# 2. judul mengandung "PN DENPASAR" atau "Pid.Sus" atau "Pid.An" (domain anak Denpasar)
def relevan(item):
    judul = item.get("judul", "").upper()
    status = item.get("status_pdf", "")
    if status != "berhasil":
        return False
    # Harus dari PN Denpasar
    if "PN DENPASAR" not in judul and "PN.DPS" not in judul:
        return False
    return True

daftar_valid = [d for d in daftar_dengan_pdf if relevan(d)]
print(f"Total setelah filter : {len(daftar_valid)}")
print(f"Dibuang              : {len(daftar_dengan_pdf) - len(daftar_valid)}")

print(f"\nDokumen yang lolos filter:")
for d in daftar_valid:
    print(f"  [{d['case_id']}] {d['judul'][:60]}")

Total sebelum filter : 100
Total setelah filter : 43
Dibuang              : 57

Dokumen yang lolos filter:
  [case_047] Putusan PN DENPASAR Nomor 37/Pid.Sus-Anak/2016/PN Dps
  [case_048] Putusan PN DENPASAR Nomor 14/Pid.Sus-Anak/2015/PN Dps
  [case_049] Putusan PN DENPASAR Nomor 3/Pid.Sus-Anak/2016/PN Dps
  [case_050] Putusan PN DENPASAR Nomor 45/Pid.An/2012/PN. Dps
  [case_051] Putusan PN DENPASAR Nomor 12/Pid.Sus.Anak/2017/PN Dps
  [case_052] Putusan PN DENPASAR Nomor 764/Pid.Sus/2014/PN Dps
  [case_053] Putusan PN DENPASAR Nomor 13/Pid.Sus.Anak/2016/PN Dps.
  [case_054] Putusan PN DENPASAR Nomor 7/Pid.Sus-Anak/2014/PN Dps
  [case_055] Putusan PN DENPASAR Nomor 34/Pid.Sus-Anak/2016/PN.Dps
  [case_057] Putusan PN DENPASAR Nomor 16 / Pid.Sus-Anak / 2016 / PN.Dps.
  [case_058] Putusan PN DENPASAR Nomor 28/Pid.Sus-Anak/2016/PN Dps
  [case_060] Putusan PN DENPASAR Nomor 3 / Pid.Sus. Anak / 2014 /PN Dps
  [case_062] Putusan PN DENPASAR Nomor 15/Pid.Sus-Anak/2015/PN Dps
  [case_063] Putusan

## Cell 6 — Konversi PDF → Plain Text

In [9]:
import pytesseract
from pdf2image import convert_from_path
from PIL import Image

# Set path Tesseract dari CONFIG
if CONFIG["TESSERACT_PATH"] and Path(CONFIG["TESSERACT_PATH"]).exists():
    pytesseract.pytesseract.tesseract_cmd = CONFIG["TESSERACT_PATH"]
    print(f"✅ Tesseract path: {CONFIG['TESSERACT_PATH']}")
else:
    # Coba pakai dari PATH sistem
    import shutil
    tess = shutil.which("tesseract")
    if tess:
        print(f"✅ Tesseract ditemukan di PATH: {tess}")
    else:
        print("❌ Tesseract tidak ditemukan!")
        print("   Download & install dari:")
        print("   https://github.com/UB-Mannheim/tesseract/wiki")
        print("   Lalu set CONFIG['TESSERACT_PATH'] di Cell 2")

# Set poppler path
POPPLER_PATH = CONFIG["POPPLER_PATH"] if CONFIG["POPPLER_PATH"] else None
if POPPLER_PATH:
    print(f"✅ Poppler path: {POPPLER_PATH}")
else:
    print("✅ Poppler: pakai dari PATH sistem")

def pdf_ke_teks(path_pdf: Path) -> str | None:
    """
    Prioritas:
    1. PyMuPDF (PDF text-based)
    2. OCR (scan PDF)
    """

    # =====================================================
    # METODE 1 : PyMuPDF
    # =====================================================
    try:
        doc = fitz.open(str(path_pdf))

        pages = []

        for page in doc:
            txt = page.get_text()

            if txt and txt.strip():
                pages.append(txt)

        teks = "\n".join(pages).strip()

        if len(teks) > 500:
            logger.info(
                f"{path_pdf.name}: PyMuPDF OK ({len(teks):,} chars)"
            )
            return teks

    except Exception as e:
        logger.warning(
            f"{path_pdf.name}: PyMuPDF gagal -> {e}"
        )

    # =====================================================
    # METODE 2 : OCR FALLBACK
    # =====================================================
    logger.info(
        f"{path_pdf.name}: fallback OCR..."
    )

    try:

        kwargs = {
            "dpi": CONFIG["OCR_DPI"]
        }

        if POPPLER_PATH:
            kwargs["poppler_path"] = POPPLER_PATH

        halaman = convert_from_path(
            str(path_pdf),
            **kwargs
        )

        teks_total = []

        for img in halaman:

            teks = pytesseract.image_to_string(
                img,
                lang=CONFIG["OCR_LANG"]
            )

            if teks.strip():
                teks_total.append(teks)

        hasil = "\n\n".join(teks_total)

        if hasil.strip() and len(hasil) > 100:
            logger.info(
                f"{path_pdf.name}: OCR OK ({len(hasil):,} chars)"
            )
            return hasil

        logger.warning(
            f"{path_pdf.name}: OCR kosong"
        )
        return None

    except Exception as e:
        logger.error(
            f"{path_pdf.name}: OCR gagal -> {e}"
        )
        return None

✅ Tesseract path: C:\Program Files\Tesseract-OCR\tesseract.exe
✅ Poppler path: C:\Users\PC-LABIT2\Downloads\Release-26.02.0-0\poppler-26.02.0\Library\bin


## Cell 7 — Cleaning & Normalisasi Teks

In [10]:
class PembersihanTeks:
    """Membersihkan dan menormalisasi teks putusan pengadilan MA RI."""

    POLA_HAPUS = [
        r"^\s*\d+\s*$",
        r"halaman\s+\d+\s+(dari|of)\s+\d+",
        r"mahkamah\s+agung\s+republik\s+indonesia",
        r"direktori\s+putusan",
        r"putusan\.mahkamahagung\.go\.id",
        r"disclaimer\s*:.*",
        r"kepaniteraan\s+mahkamah\s+agung.*",
    ]

    def bersihkan(self, teks: str, lowercase: bool = False) -> str:
        """
        Pipeline pembersihan:
        1. Normalisasi Unicode (NFKC)
        2. Hapus karakter non-printable
        3. Hapus header/footer/watermark
        4. Normalisasi spasi & baris
        5. (Opsional) Lowercase
        """
        if not teks:
            return ""

        teks = unicodedata.normalize("NFKC", teks)
        teks = re.sub(r"[^\x09\x0A\x0D\x20-\x7E\x80-\xFF\u0100-\uFFFF]", " ", teks)

        baris_bersih = []
        for baris in teks.split("\n"):
            baris_lower = baris.lower().strip()
            buang = any(re.search(p, baris_lower) for p in self.POLA_HAPUS)
            if not buang:
                baris_bersih.append(baris)
        teks = "\n".join(baris_bersih)

        teks = re.sub(r"[ \t]+", " ", teks)
        teks = re.sub(r"\n{3,}", "\n\n", teks)
        teks = re.sub(r"\n\s+\n", "\n\n", teks)
        teks = teks.strip()

        if lowercase:
            teks = teks.lower()

        return teks

    def hitung_kata(self, teks: str) -> int:
        return len(teks.split()) if teks else 0

    def hitung_validitas(self, teks_asli: str, teks_bersih: str) -> float:
        """
        Validitas berdasarkan rasio KARAKTER (bukan kata).
        Lebih stabil untuk PDF scan yang teks OCR-nya tidak konsisten.
        """
        kar_asli   = len(teks_asli.replace(" ", "").replace("\n", ""))
        kar_bersih = len(teks_bersih.replace(" ", "").replace("\n", ""))
        if kar_asli == 0:
            return 0.0
        return kar_bersih / kar_asli


pembersih = PembersihanTeks()
print("✅ Kelas PembersihanTeks siap")

✅ Kelas PembersihanTeks siap


## Cell 8 — Pipeline Utama: PDF → Teks Bersih → Validasi → Simpan

In [11]:
def pipeline_tahap1(daftar: list) -> dict:
    """
    Pipeline lengkap Tahap 1:
    PDF → Teks → Cleaning → Validasi → Simpan ke /data/raw/

    Filter:
    - Dokumen dengan status_pdf 'skip_tanpa_pdf' tidak diproses
    - Validasi: MIN_KATA dan MIN_VALIDITAS dari CONFIG
    """
    stats = {
        "total"          : 0,
        "berhasil"       : 0,
        "gagal_ekstrak"  : 0,
        "gagal_validasi" : 0,
        "detail"         : [],
    }

    # Hanya proses yang PDF-nya berhasil
    valid_untuk_proses = [
        p for p in daftar
        if p.get("status_pdf") in ("berhasil", "sudah_ada") and p.get("path_pdf")
    ]

    print(f"🔄 Pipeline: {len(valid_untuk_proses)} PDF akan diproses")
    print(f"   (Dilewati: dokumen tanpa PDF tidak diproses)\n")
    logger.info(f"Pipeline mulai: {len(valid_untuk_proses)} PDF")

    for putusan in tqdm(valid_untuk_proses, desc="Cleaning"):
        case_id  = putusan["case_id"]
        path_pdf = Path(putusan["path_pdf"])
        path_txt = DATA_RAW / f"{case_id}.txt"
        stats["total"] += 1

        info = {
            "case_id"    : case_id,
            "path_txt"   : str(path_txt),
            "jumlah_kata": 0,
            "validitas"  : 0.0,
            "status"     : "",
        }

        # Skip jika teks sudah ada
        if path_txt.exists() and path_txt.stat().st_size > 100:
            teks_ada = path_txt.read_text(encoding="utf-8", errors="ignore")
            info["jumlah_kata"] = pembersih.hitung_kata(teks_ada)
            info["status"]      = "sudah_ada"
            stats["berhasil"]  += 1
            logger.info(f"{case_id}: teks sudah ada, skip")
            stats["detail"].append(info)
            continue

        # Konversi PDF → teks mentah
        teks_mentah = pdf_ke_teks(path_pdf)
        if not teks_mentah:
            info["status"] = "gagal_ekstrak"
            stats["gagal_ekstrak"] += 1
            logger.error(f"{case_id}: ekstraksi teks gagal atau kosong")
            stats["detail"].append(info)
            continue

        # Cleaning
        teks_bersih = pembersih.bersihkan(teks_mentah, lowercase=False)

        # Validasi jumlah kata
        jumlah_kata = pembersih.hitung_kata(teks_bersih)
        validitas   = pembersih.hitung_validitas(teks_mentah, teks_bersih)

        info["jumlah_kata"] = jumlah_kata
        info["validitas"]   = round(validitas, 3)

        if jumlah_kata < CONFIG["MIN_KATA"]:
            info["status"] = "gagal_validasi_kata"
            stats["gagal_validasi"] += 1
            logger.warning(f"{case_id}: hanya {jumlah_kata} kata (min {CONFIG['MIN_KATA']})")
            stats["detail"].append(info)
            continue

        # Validasi rasio karakter
        if validitas < CONFIG["MIN_VALIDITAS"]:
            info["status"] = "gagal_validasi_rasio"
            stats["gagal_validasi"] += 1
            logger.warning(f"{case_id}: validitas karakter {validitas:.2%} < {CONFIG['MIN_VALIDITAS']:.0%} — PDF mungkin corrupt atau kosong")
            stats["detail"].append(info)
            continue

        # Simpan teks bersih
        path_txt.write_text(teks_bersih, encoding="utf-8")
        info["status"] = "berhasil"
        stats["berhasil"] += 1
        logger.info(f"{case_id}: ✅ {jumlah_kata} kata, validitas={validitas:.2%}")
        stats["detail"].append(info)

    return stats

print("✅ Fungsi pipeline_tahap1 siap")

✅ Fungsi pipeline_tahap1 siap


In [12]:
# ============================================================
# HAPUS FILE .txt LAMA sebelum jalankan pipeline OCR
# Diperlukan agar pipeline tidak skip file disclaimer lama
# ============================================================
file_txt_lama = sorted(DATA_RAW.glob("*.txt"))
if file_txt_lama:
    print(f"\U0001f5d1\ufe0f  Menghapus {len(file_txt_lama)} file .txt lama (isi disclaimer)...")
    for f in file_txt_lama:
        f.unlink()
    print(f"\u2705 {len(file_txt_lama)} file dihapus — folder data/raw/ siap untuk OCR")
else:
    print("\u2705 Folder data/raw/ sudah kosong, siap untuk OCR")

🗑️  Menghapus 39 file .txt lama (isi disclaimer)...
✅ 39 file dihapus — folder data/raw/ siap untuk OCR


In [13]:
# Jalankan pipeline
statistik = pipeline_tahap1(daftar_valid)

# Simpan statistik
stats_file = LOGS_DIR / "stats_tahap1.json"
with open(stats_file, "w", encoding="utf-8") as f:
    json.dump(statistik, f, ensure_ascii=False, indent=2)

print(f"\n📊 Statistik Pipeline Tahap 1:")
print(f"   Total diproses   : {statistik['total']}")
print(f"   ✅ Berhasil       : {statistik['berhasil']}")
print(f"   ❌ Gagal ekstrak  : {statistik['gagal_ekstrak']}")
print(f"   ⚠️  Gagal validasi : {statistik['gagal_validasi']}")
print(f"\n💾 Statistik: {stats_file}")

2026-06-13 22:16:31,976 | INFO | Pipeline mulai: 43 PDF


🔄 Pipeline: 43 PDF akan diproses
   (Dilewati: dokumen tanpa PDF tidak diproses)



Cleaning:   0%|          | 0/43 [00:00<?, ?it/s]2026-06-13 22:16:32,025 | INFO | case_047.pdf: PyMuPDF OK (44,913 chars)
2026-06-13 22:16:32,039 | INFO | case_047: ✅ 5499 kata, validitas=81.90%
2026-06-13 22:16:32,159 | INFO | case_048.pdf: PyMuPDF OK (179,274 chars)
2026-06-13 22:16:32,187 | INFO | case_048: ✅ 21840 kata, validitas=84.20%
Cleaning:   5%|▍         | 2/43 [00:00<00:04,  9.63it/s]2026-06-13 22:16:32,260 | INFO | case_049.pdf: PyMuPDF OK (108,208 chars)
2026-06-13 22:16:32,278 | INFO | case_049: ✅ 12476 kata, validitas=81.77%
2026-06-13 22:16:32,349 | INFO | case_050.pdf: PyMuPDF OK (106,929 chars)
2026-06-13 22:16:32,370 | INFO | case_050: ✅ 11032 kata, validitas=81.60%
Cleaning:   9%|▉         | 4/43 [00:00<00:03, 10.33it/s]2026-06-13 22:16:32,427 | INFO | case_051.pdf: PyMuPDF OK (68,683 chars)
2026-06-13 22:16:32,439 | INFO | case_051: ✅ 6691 kata, validitas=85.43%
2026-06-13 22:16:32,495 | INFO | case_052.pdf: PyMuPDF OK (80,387 chars)
2026-06-13 22:16:32,511 | INFO 


📊 Statistik Pipeline Tahap 1:
   Total diproses   : 43
   ✅ Berhasil       : 43
   ❌ Gagal ekstrak  : 0
   ⚠️  Gagal validasi : 0

💾 Statistik: D:\amad\Paper\Tugas\PK\Penalaran-Komputer-Sub-3\STEP 1\logs\stats_tahap1.json


## Cell 9 — Validasi Akhir & Laporan

In [14]:
def validasi_akhir() -> None:
    """Cek hasil akhir dan pastikan kuota minimum terpenuhi."""
    file_txt = sorted(DATA_RAW.glob("*.txt"))
    total    = len(file_txt)

    print("=" * 55)
    print("       LAPORAN VALIDASI AKHIR — TAHAP 1")
    print("=" * 55)
    print(f"  Direktori raw    : {DATA_RAW}")
    print(f"  Total file .txt  : {total}")
    print(f"  Minimum required : {CONFIG['MIN_DOKUMEN']}")

    if total >= CONFIG["MIN_DOKUMEN"]:
        print(f"  Status           : ✅ MEMENUHI KUOTA")
        logger.info(f"Validasi akhir: LULUS — {total} dokumen")
    else:
        kekurangan = CONFIG["MIN_DOKUMEN"] - total
        print(f"  Status           : ❌ KURANG {kekurangan} dokumen")
        logger.warning(f"Validasi akhir: KURANG {kekurangan} dokumen")

    if file_txt:
        kata_per_file = [
            len(f.read_text(encoding="utf-8", errors="ignore").split())
            for f in file_txt
        ]
        print(f"\n  Statistik jumlah kata per dokumen:")
        print(f"    Min  : {min(kata_per_file):,}")
        print(f"    Max  : {max(kata_per_file):,}")
        print(f"    Rata : {sum(kata_per_file) // len(kata_per_file):,}")

    print("=" * 55)
    print(f"  Log lengkap      : {LOGS_DIR / 'cleaning.log'}")
    print("=" * 55)

    print("\n  Contoh file yang dihasilkan:")
    for f in file_txt[:5]:
        ukuran = f.stat().st_size / 1024
        print(f"    {f.name}  ({ukuran:.1f} KB)")

validasi_akhir()

2026-06-13 22:16:34,993 | WARNING | Validasi akhir: KURANG 7 dokumen


       LAPORAN VALIDASI AKHIR — TAHAP 1
  Direktori raw    : D:\amad\Paper\Tugas\PK\Penalaran-Komputer-Sub-3\STEP 1\data\raw
  Total file .txt  : 43
  Minimum required : 50
  Status           : ❌ KURANG 7 dokumen

  Statistik jumlah kata per dokumen:
    Min  : 62
    Max  : 21,840
    Rata : 7,533
  Log lengkap      : D:\amad\Paper\Tugas\PK\Penalaran-Komputer-Sub-3\STEP 1\logs\cleaning.log

  Contoh file yang dihasilkan:
    case_047.txt  (36.9 KB)
    case_048.txt  (150.6 KB)
    case_049.txt  (88.5 KB)
    case_050.txt  (88.1 KB)
    case_051.txt  (57.8 KB)


## Cell 10 — Preview Hasil Cleaning

In [15]:
file_contoh = sorted(DATA_RAW.glob("*.txt"))

if file_contoh:
    teks_preview = file_contoh[0].read_text(encoding="utf-8", errors="ignore")
    print(f"📄 Preview: {file_contoh[0].name}")
    print("-" * 60)
    print(teks_preview[:800])
    print("-" * 60)
    print(f"... (total {len(teks_preview.split())} kata)")
else:
    print("⚠️  Belum ada file .txt — jalankan pipeline terlebih dahulu")

📄 Preview: case_047.txt
------------------------------------------------------------
hkama
ahkamah Agung Repub
ahkamah Agung Republik Indonesia
mah Agung Republik Indonesia
blik Indonesi
P U T U S A N
Nomor 37/Pid.Sus-Anak/2016/PN Dps
DEMI KEADILAN BERDASARKAN KETUHANAN YANG MAHA ESA
Pengadilan Negeri Denpasar yang menerima, memeriksa dan mengadili 
perkara-perkara pidana sidang anak pada peradilan tingkat pertama oleh Hakim 
Tunggal telah menjatuhkan putusan sebagai berikut dalam perkara atas nama 
Anak :
Nama lengkap
: TERDAKWA ANAK ;
Tempat lahir
: Denpasar ;
Umur / Tgl lahir
: 15 Tahun / 16 Mei 2001 ;
Jenis Kelamin
: Laki-laki ;
Kebangsaan
: Indonesia ;
Tempat tinggal
: Denpasar ;
Agama
: Islam ;
Pekerjaan
: Tidak ada ;
Pendidikan 
: SMP ( tidak tamat ) ;
Anak ditahan :
•
Penyidik, tidak ditahan ;
•
Penuntut Umum, sejak tanggal 6 Desember 2016 sampai dengan tanggal 10 
------------------------------------------------------------
... (total 5499 kata)


---
## ✅ Tahap 1 Selesai!

**Struktur output yang dihasilkan:**
```
/data/
├── raw/
│   ├── case_001.txt
│   ├── case_002.txt
│   └── ...
├── pdf/
│   ├── case_001.pdf
│   └── ...
├── metadata_raw.json
└── metadata_with_pdf.json
/logs/
├── cleaning.log
└── stats_tahap1.json
```

**Langkah selanjutnya:** Buka `02_case_representation.ipynb` untuk Tahap 2.

In [16]:
from pathlib import Path

print("Current Working Directory:")
print(Path.cwd())

print("\nSemua cases.csv yang ditemukan:")
for p in Path.cwd().rglob("cases.csv"):
    print(p.resolve())

print("\nSemua cases.json yang ditemukan:")
for p in Path.cwd().rglob("cases.json"):
    print(p.resolve())

Current Working Directory:
d:\amad\Paper\Tugas\PK\Penalaran-Komputer-Sub-3\STEP 1

Semua cases.csv yang ditemukan:
D:\amad\Paper\Tugas\PK\Penalaran-Komputer-Sub-3\STEP 1\data\processed\cases.csv

Semua cases.json yang ditemukan:
D:\amad\Paper\Tugas\PK\Penalaran-Komputer-Sub-3\STEP 1\data\processed\cases.json
